# exp-013: On-Policy Rollout Generation

Generate N rollouts per training problem using the 0.85 adapter.
Verify with rule solver + self-consistency voting for symbol family.
Output: on-policy training data for OPD self-improvement.

In [ ]:
# Install vllm and dependencies from local wheels
import subprocess, sys, glob, os, shutil, zipfile, re

# 1. Install msgspec first (required by vllm)
msgspec_candidates = sorted(glob.glob("/kaggle/input/**/msgspec*.whl", recursive=True))
print("Found msgspec wheels:", msgspec_candidates)
if msgspec_candidates:
    msgspec_wheel = msgspec_candidates[-1]  # latest version
    print(f"Installing msgspec from: {msgspec_wheel}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", msgspec_wheel], check=True)
else:
    print("Warning: no msgspec wheel found, vllm may fail")

# 2. Install vllm — prefer 0.20.1 (better model support), fix metadata if needed
vllm_candidates = sorted(glob.glob("/kaggle/input/**/vllm*.whl", recursive=True))
print("Found vllm wheels:", vllm_candidates)

if not vllm_candidates:
    raise FileNotFoundError("No vllm wheel found under /kaggle/input")

vllm_wheel = None
for w in vllm_candidates:
    if "0.20.1" in w:
        # Fix malformed version "0.20.1cu129" -> "0.20.1+cu129"
        print(f"Fixing 0.20.1 wheel metadata: {w}")
        fixed_dir = "/tmp/vllm_fixed"
        if os.path.exists(fixed_dir):
            shutil.rmtree(fixed_dir)
        os.makedirs(fixed_dir)
        with zipfile.ZipFile(w, 'r') as zf:
            zf.extractall(fixed_dir)
        for root, dirs, files in os.walk(fixed_dir):
            for f in files:
                if f == 'METADATA':
                    meta_path = os.path.join(root, f)
                    with open(meta_path, 'r') as fh:
                        content = fh.read()
                    content = re.sub(r'^Version:\s*0\.20\.1cu129', 'Version: 0.20.1+cu129', content, flags=re.MULTILINE)
                    with open(meta_path, 'w') as fh:
                        fh.write(content)
                    dist_info_dir = os.path.dirname(meta_path)
                    if '0.20.1cu129' in dist_info_dir:
                        new_dir = dist_info_dir.replace('0.20.1cu129', '0.20.1+cu129')
                        os.rename(dist_info_dir, new_dir)
        fixed_whl = "/tmp/vllm-0.20.1+cu129-cp38-abi3-manylinux_2_31_x86_64.whl"
        with zipfile.ZipFile(fixed_whl, 'w', zipfile.ZIP_DEFLATED) as zf:
            for root, dirs, files in os.walk(fixed_dir):
                for f in files:
                    full = os.path.join(root, f)
                    arcname = os.path.relpath(full, fixed_dir)
                    zf.write(full, arcname)
        vllm_wheel = fixed_whl
        break

# Fallback to 0.18.0
if vllm_wheel is None:
    for w in vllm_candidates:
        if "0.18.0" in w:
            vllm_wheel = w
            break

if vllm_wheel is None:
    raise FileNotFoundError("No usable vllm wheel found")

print(f"Installing vllm from: {vllm_wheel}")
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", vllm_wheel], check=True)

# 3. Install peft
peft_candidates = glob.glob("/kaggle/input/**/peft*.whl", recursive=True)
if peft_candidates:
    print(f"Installing peft from: {peft_candidates[0]}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", peft_candidates[0]], check=True)
else:
    packages_dir = "/kaggle/input/nemotron-packages/packages"
    if os.path.isdir(packages_dir):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", packages_dir, "peft"], check=True)

print("All packages installed.")

In [ ]:
import torch
print(f"Python: {sys.version}")
print(f"Torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

In [ ]:
# Configuration
import glob, os

NUM_ROLLOUTS = 3
TEMPERATURE = 0.6
TOP_P = 0.95
MAX_TOKENS = 7680
GPU_MEMORY_UTILIZATION = 0.85

# Find adapter path
adapter_candidates = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
print("Found adapter configs:", adapter_candidates)
if adapter_candidates:
    ADAPTER_PATH = os.path.dirname(adapter_candidates[0])
else:
    ADAPTER_PATH = '/kaggle/input/trained-adapter'

print(f'Config: {NUM_ROLLOUTS} rollouts, temp={TEMPERATURE}, max_tokens={MAX_TOKENS}')
print(f'Adapter: {ADAPTER_PATH}')

In [ ]:
import kagglehub

MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
print(f'Model path: {MODEL_PATH}')

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

# Initialize vLLM with LoRA support
llm = LLM(
    model=str(MODEL_PATH),
    tensor_parallel_size=1,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    dtype='auto',
    max_model_len=8192,
    trust_remote_code=True,
    enable_lora=True,
    max_lora_rank=32,
    enable_prefix_caching=True,
)

tokenizer = llm.get_tokenizer()
print('vLLM initialized with LoRA support.')

In [ ]:
import math
import re
import json

# Answer extraction (from huikang notebook)
def extract_final_answer(text):
    if text is None:
        return 'NOT_FOUND'
    matches = re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()
    patterns = [
        r'The final answer is:\s*([^\n]+)',
        r'Final answer is:\s*([^\n]+)',
        r'Final answer\s*[:：]\s*([^\n]+)',
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].strip()
    matches = re.findall(r'-?\d+(?:\.\d+)?', text)
    if matches:
        return matches[-1]
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else 'NOT_FOUND'

# Verification (from huikang notebook)
def verify(stored_answer, predicted):
    stored_answer = str(stored_answer).strip()
    predicted = str(predicted).strip()
    try:
        stored_num = float(stored_answer)
        predicted_num = float(predicted)
        return math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored_answer.lower()

print('Helper functions loaded.')

In [ ]:
# Load training data
import pandas as pd

# Find train.csv — prioritize competition path
COMPETITION_CSV = '/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv'
if os.path.exists(COMPETITION_CSV):
    TRAIN_CSV = COMPETITION_CSV
else:
    # Fallback: find any train.csv, but prefer competition-related ones
    train_csv_candidates = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    print("Found train.csv files:", train_csv_candidates)
    if not train_csv_candidates:
        raise FileNotFoundError("No train.csv found")
    TRAIN_CSV = train_csv_candidates[0]

train_df = pd.read_csv(TRAIN_CSV)
print(f'Loaded {len(train_df)} training problems from {TRAIN_CSV}')
print(f'Columns: {list(train_df.columns)}')

# Family classification
def classify_family(prompt):
    p = prompt.lower()
    if 'bit manipulation' in p or '8-bit binary' in p:
        return 'bit'
    if 'encryption' in p or 'cipher' in p:
        return 'cipher'
    if 'roman' in p:
        return 'roman'
    if 'gravity' in p or 'fallen from' in p or 'dropped from' in p:
        return 'gravity'
    if 'unit' in p or 'convert' in p or 'measurement' in p:
        return 'unit'
    if 'symbol' in p or ('in alice' in p and 'symbolic' in p):
        return 'symbol'
    return 'unknown'

train_df['family'] = train_df['prompt'].apply(classify_family)
print(f'\nFamily distribution:')
print(train_df['family'].value_counts())

In [ ]:
# Format prompts
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

prompts = []
for _, row in train_df.iterrows():
    user_content = row['prompt'] + PROMPT_SUFFIX
    try:
        prompt = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': user_content}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
    except Exception:
        prompt = user_content
    prompts.append(prompt)

print(f'Formatted {len(prompts)} prompts')

In [ ]:
# Generate rollouts with vLLM (fast batched inference)
import time

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS,
)

all_rollout_results = []

for rollout_idx in range(NUM_ROLLOUTS):
    print(f'\n--- Rollout {rollout_idx + 1}/{NUM_ROLLOUTS} ---')
    t0 = time.time()

    # vLLM processes all prompts in parallel — much faster
    outputs = llm.generate(
        prompts,
        sampling_params=sampling_params,
        lora_request=LoRARequest('adapter', 1, ADAPTER_PATH),
    )

    elapsed = time.time() - t0
    print(f'  Generated {len(outputs)} outputs in {elapsed:.1f}s ({elapsed/60:.1f}min)')

    for i, (idx, row) in enumerate(train_df.iterrows()):
        raw_text = outputs[i].outputs[0].text
        predicted = extract_final_answer(raw_text)
        correct = verify(str(row['answer']), predicted)

        all_rollout_results.append({
            'id': row['id'],
            'family': row['family'],
            'prompt': row['prompt'],
            'answer': str(row['answer']),
            'rollout_idx': rollout_idx,
            'raw_output': raw_text,
            'predicted': predicted,
            'correct': correct,
        })

    rollout_df = pd.DataFrame([r for r in all_rollout_results if r['rollout_idx'] == rollout_idx])
    acc = rollout_df['correct'].mean()
    print(f'  Accuracy: {acc:.4f} ({rollout_df["correct"].sum()}/{len(rollout_df)})')

print(f'\nTotal rollout results: {len(all_rollout_results)}')

In [ ]:
# Build per-problem analysis
from collections import Counter

results_df = pd.DataFrame(all_rollout_results)

problem_summary = []
for problem_id in train_df['id'].unique():
    problem_rollouts = results_df[results_df['id'] == problem_id]
    family = problem_rollouts.iloc[0]['family']
    answer = problem_rollouts.iloc[0]['answer']

    num_correct = problem_rollouts['correct'].sum()
    num_total = len(problem_rollouts)

    predicted_answers = problem_rollouts['predicted'].tolist()
    answer_counts = Counter(predicted_answers)
    majority_answer, majority_count = answer_counts.most_common(1)[0]

    majority_correct = verify(str(answer), majority_answer)

    problem_summary.append({
        'id': problem_id,
        'family': family,
        'answer': answer,
        'num_correct': num_correct,
        'num_total': num_total,
        'accuracy': num_correct / num_total,
        'majority_answer': majority_answer,
        'majority_count': majority_count,
        'majority_correct': majority_correct,
    })

summary_df = pd.DataFrame(problem_summary)
print(f'Per-problem summary: {len(summary_df)} problems')
print(f'\nOverall accuracy (single rollout): {results_df["correct"].mean():.4f}')
print(f'Self-consistency accuracy: {summary_df["majority_correct"].mean():.4f}')

In [ ]:
# Per-family breakdown
print('=== Per-Family Breakdown ===')
for family in sorted(summary_df['family'].unique()):
    fam = summary_df[summary_df['family'] == family]
    single_acc = results_df[results_df['family'] == family]['correct'].mean()
    majority_acc = fam['majority_correct'].mean()
    print(f'{family:20s}: single={single_acc:.4f}  majority={majority_acc:.4f}  n={len(fam)}')

In [ ]:
# Build on-policy training dataset
on_policy_records = []

for _, row in summary_df.iterrows():
    problem_id = row['id']
    family = row['family']
    answer = row['answer']
    
    problem_rollouts = results_df[results_df['id'] == problem_id]
    
    if family == 'symbol':
        target_answer = row['majority_answer']
        best_rollout = problem_rollouts[problem_rollouts['predicted'] == target_answer].iloc[0]
        on_policy_records.append({
            'id': problem_id,
            'family': family,
            'prompt': best_rollout['prompt'],
            'answer': target_answer,
            'cot': best_rollout['raw_output'],
            'source': 'self_consistency',
            'verified': row['majority_correct'],
        })
    else:
        correct_rollouts = problem_rollouts[problem_rollouts['correct'] == True]
        if len(correct_rollouts) > 0:
            best = correct_rollouts.iloc[0]
            on_policy_records.append({
                'id': problem_id,
                'family': family,
                'prompt': best['prompt'],
                'answer': answer,
                'cot': best['raw_output'],
                'source': 'on_policy_correct',
                'verified': True,
            })

on_policy_df = pd.DataFrame(on_policy_records)
print(f'\nOn-policy training records: {len(on_policy_df)}')
print(f'Source distribution:')
print(on_policy_df['source'].value_counts())
print(f'\nFamily distribution:')
print(on_policy_df['family'].value_counts())

In [ ]:
# Save results
results_df.to_csv('/kaggle/working/rollout_results.csv', index=False)
summary_df.to_csv('/kaggle/working/problem_summary.csv', index=False)
on_policy_df.to_csv('/kaggle/working/on_policy_data.csv', index=False)

print('Saved:')
print(f'  rollout_results.csv: {len(results_df)} rows (all rollouts)')
print(f'  problem_summary.csv: {len(summary_df)} rows (per-problem)')
print(f'  on_policy_data.csv:  {len(on_policy_df)} rows (training data)')

In [ ]:
# Summary statistics
print('=== Final Summary ===')
print(f'Total problems: {len(summary_df)}')
print(f'Single rollout accuracy: {results_df["correct"].mean():.4f}')
print(f'Self-consistency accuracy: {summary_df["majority_correct"].mean():.4f}')
print(f'On-policy training records: {len(on_policy_df)}')
print(f'  - Verified correct: {on_policy_df["verified"].sum()}')
print(f'  - Self-consistency (symbol): {len(on_policy_df[on_policy_df["source"] == "self_consistency"])}')